In [1]:
from load_dotenv import load_dotenv

load_dotenv()

True

# CUSTOM TOOL

In [2]:
from langchain.tools import tool
from random import randint

@tool(
    name_or_callable="cofrinhos_extrato",
    description="ferramenta para extrair extrato dos Cofrinhos, para verificar se existe fundos disponíveis."
)
def cofrinhos_extrato() -> str:
    saldo = randint(0, 10000)
    return f"Encontrado R$ {saldo},00 nos Cofrinhos do cliente"

@tool(
    name_or_callable="cofrinhos_recuperar",
    description="ferramenta para recuperar fundos dos Cofrinhos."
)
def cofrinhos_recuperar(valor: str) -> str:
    return f"Recuperado {valor} dos Cofrinhos do cliente"

@tool(
    name_or_callable="execute_pix",
    description="ferramenta para executar um PIX para qualquer pessoa e qualquer valor."
)
def execute_pix(destinatario: str, valor: str) -> str:
    return f"PIX de {valor} enviado para {destinatario}"

### testando as ferramentas

In [3]:
extract = cofrinhos_extrato.invoke("")
recuperar = cofrinhos_recuperar.invoke("R$ 500,00")
pix = execute_pix.invoke({"destinatario": "João", "valor": "R$ 500,00"})

print(extract)
print(recuperar)
print(pix)


Encontrado R$ 6850,00 nos Cofrinhos do cliente
Recuperado R$ 500,00 dos Cofrinhos do cliente
PIX de R$ 500,00 enviado para João


In [4]:
tools = [cofrinhos_extrato, cofrinhos_recuperar, execute_pix]

# Planner

In [5]:
from typing import Sequence

from langchain_core.language_models import BaseChatModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch
from langchain_core.tools import BaseTool
from langchain_core.messages import (
    BaseMessage,
    FunctionMessage,
    HumanMessage,
    SystemMessage,
)

from src.llm_compiler.output_parser import LLMCompilerPlanParser, Task
from langchain_openai import ChatOpenAI

In [6]:
system_prompt = """Dada uma consulta do usuário, crie um plano para resolvê-la. Cada plano deve ser composto por uma ação entre os seguintes {num_tools} tipos:
{tool_descriptions}
{num_tools}. join(): Coleta e combina resultados de ações anteriores.

 - Um agente LLM é chamado ao invocar join() para finalizar a consulta do usuário ou aguardar até que os planos sejam executados.
 - join deve ser sempre a última ação do plano e será chamada em dois cenários:
  (a) se a resposta puder ser determinada reunindo as saídas das tarefas para gerar a resposta final.
  (b) se a resposta não puder ser determinada na fase de planejamento antes de executar os planos. Diretrizes:
 - Cada ação descrita acima contém tipos de entrada/saída e descrição.
   - Você deve aderir estritamente aos tipos de entrada e saída de cada ação.
   - As descrições das ações contêm diretrizes. Você DEVE seguir estritamente essas diretrizes ao usar as ações.
 - Cada ação no plano deve ser estritamente um dos tipos acima. Siga as convenções de Python para cada ação.
 - Cada ação DEVE ter um ID único, estritamente crescente.
 - As entradas das ações podem ser constantes ou saídas de ações anteriores. No segundo caso, use o formato $id para indicar o ID da ação anterior cuja saída será a entrada.
 - Sempre chame join como a última ação do plano. Escreva '<END_OF_PLAN>' depois de chamar join
 - Garanta que o plano maximize a paralelização.
 - Use apenas os tipos de ação fornecidos. Se uma consulta não puder ser atendida com eles, invoque a ação join para os próximos passos.
 - Nunca introduza novas ações além das fornecidas.

FORMATO OBRIGATÓRIO DE SAÍDA — siga EXATAMENTE este padrão de texto simples, sem JSON, sem markdown:

Thought: <raciocínio opcional>
1. nome_da_ferramenta("argumento1", "argumento2")
2. outra_ferramenta($1)
3. join()
<END_OF_PLAN>"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("placeholder", "{messages}")
])
print(prompt.pretty_print())


================================ System Message ================================

Dada uma consulta do usuário, crie um plano para resolvê-la. Cada plano deve ser composto por uma ação entre os seguintes {num_tools} tipos:
{tool_descriptions}
{num_tools}. join(): Coleta e combina resultados de ações anteriores.

 - Um agente LLM é chamado ao invocar join() para finalizar a consulta do usuário ou aguardar até que os planos sejam executados.
 - join deve ser sempre a última ação do plano e será chamada em dois cenários:
  (a) se a resposta puder ser determinada reunindo as saídas das tarefas para gerar a resposta final.
  (b) se a resposta não puder ser determinada na fase de planejamento antes de executar os planos. Diretrizes:
 - Cada ação descrita acima contém tipos de entrada/saída e descrição.
   - Você deve aderir estritamente aos tipos de entrada e saída de cada ação.
   - As descrições das ações contêm diretrizes. Você DEVE seguir estritamente essas diretrizes ao usar as ações.
 

In [7]:
def create_planner(
    llm: BaseChatModel, tools: Sequence[BaseTool], base_prompt: ChatPromptTemplate
):
    def _format_tool(i: int, tool) -> str:
        args_str = ", ".join(
            f"{k}: {v.get('type', 'str')}"
            for k, v in (tool.args or {}).items()
        )
        return f"{i+1}. {tool.name}({args_str}) - {tool.description}\n"

    tool_descriptions = "\n".join(
        _format_tool(i, tool)
        for i, tool in enumerate(tools)
    )
    planner_prompt = base_prompt.partial(
        replan="",
        num_tools=len(tools)
        + 1,  # Add one because we're adding the join() tool at the end.
        tool_descriptions=tool_descriptions,
    )
    replanner_prompt = base_prompt.partial(
        replan=' - Você recebe um "Plano Anterior" que é o plano criado pelo agente anterior junto com os resultados da execução '
        "(fornecidos como Observação) de cada plano e um pensamento geral (fornecido como Pensamento) sobre os resultados executados."
        'Você DEVE usar essas informações para criar o próximo plano sob "Plano Atual".\n'
        ' - Ao iniciar o Plano Atual, você deve começar com um "Pensamento" que descreve a estratégia para o próximo plano.\n'
        " - No Plano Atual, você NUNCA deve repetir as ações que já foram executadas no Plano Anterior.\n"
        " - Você deve continuar o índice de tarefas a partir do final do anterior. Não repita índices de tarefas.",
        num_tools=len(tools) + 1,
        tool_descriptions=tool_descriptions,
    )

    def should_replan(state: list):
        # Context is passed as a system message
        return isinstance(state[-1], SystemMessage)

    def wrap_messages(state: list):
        return {"messages": state}

    def wrap_and_get_last_index(state: list):
        next_task = 0
        for message in state[::-1]:
            if isinstance(message, FunctionMessage):
                next_task = message.additional_kwargs["idx"] + 1
                break
        state[-1].content = state[-1].content + f" - Begin counting at : {next_task}"
        return {"messages": state}

    return (
        RunnableBranch(
            (should_replan, wrap_and_get_last_index | replanner_prompt),
            wrap_messages | planner_prompt,
        )
        | llm
        | LLMCompilerPlanParser(tools=tools)
    )


In [8]:
llm = ChatOpenAI(model="gpt-5.2")
# This is the primary "agent" in our application
planner = create_planner(llm, tools, prompt)

In [9]:
example_question = "quero fazer um regaste dos cofrinhos de 500 reais e depois com dinheiro fazer uma transferência de pix de 100 reais para Leonardo"

# Mostra o plano bruto gerado pelo LLM (antes do parsing)
raw_output = (planner.steps[0] | planner.steps[1]).invoke([HumanMessage(content=example_question)])
print("=" * 50)
print("PLANO GERADO PELO LLM:")
print("=" * 50)
print(raw_output.content)

# Mostra o plano parseado com ordem de execução e dependências
print("=" * 50)
print("ORDEM DE EXECUÇÃO:")
print("=" * 50)
for graph_dict in planner.stream([HumanMessage(content=example_question)]):
    for idx, task in sorted(graph_dict.items()):
        deps = f"depende de {task.dependencies}" if task.dependencies else "sem dependências"
        print(f"[{idx}] {task.name}{task.args}  ({deps})")
    print()

print("=" * 50)
print("EXECUÇÃO DAS TOOLS:")
print("=" * 50)
for graph_dict in planner.stream([HumanMessage(content=example_question)]):
    for task in graph_dict.values():
        tool_obj = next((t for t in tools if t.name == task.name), None)
        if tool_obj:
            arg_keys = list(tool_obj.args.keys())
            args_dict = dict(zip(arg_keys, task.args))
            print(tool_obj, args_dict)
            result = tool_obj.invoke(args_dict)
            print(f"Resultado: {result}")
        else:
            print(task.name, task.args)
        print("---")


PLANO GERADO PELO LLM:
Thought: Verificar saldo nos cofrinhos, resgatar 500 e então fazer um PIX de 100 para Leonardo.
1. cofrinhos_extrato()
2. cofrinhos_recuperar("500")
3. execute_pix("Leonardo", "100")
4. join()
<END_OF_PLAN>
ORDEM DE EXECUÇÃO:
[1] cofrinhos_extrato()  (sem dependências)
[2] cofrinhos_recuperar('500',)  (sem dependências)
[3] execute_pix('Leonardo', '100')  (sem dependências)
[4] join()  (depende de [1, 2, 3])

EXECUÇÃO DAS TOOLS:
name='cofrinhos_extrato' description='ferramenta para extrair extrato dos Cofrinhos, para verificar se existe fundos disponíveis.' args_schema=<class 'langchain_core.utils.pydantic.cofrinhos_extrato'> func=<function cofrinhos_extrato at 0x1086fe0c0> {}
Resultado: Encontrado R$ 4477,00 nos Cofrinhos do cliente
---
name='cofrinhos_recuperar' description='ferramenta para recuperar fundos dos Cofrinhos.' args_schema=<class 'langchain_core.utils.pydantic.cofrinhos_recuperar'> func=<function cofrinhos_recuperar at 0x10d7cd8a0> {'valor': '500'}


# Task Fetching Unit